# VOZ-HSD - Baseline XLM-RoBERTa + ViCLSR

Notebook Kaggle chạy hai baseline `FacebookAI/xlm-roberta-base` và `huynhtin/ViCLSR`.

Bật Internet và GPU T4 trước khi chạy. ViCLSR dùng XLM-RoBERTa-Large nên chậm và tốn VRAM hơn XLM-RoBERTa-base.

In [ ]:
import os
import subprocess
from pathlib import Path

EXP_REL = Path("notebooks/models/baselines/VOZ-HSD - Baseline XLM-RoBERTa_ViCLSR")
ROOT = Path.cwd()

# If this notebook is opened outside the repo, clone the project first.
if not (ROOT / EXP_REL / "run_two_models.py").exists():
    if not (ROOT / "ViAmpleHate").exists():
        subprocess.run(["git", "clone", "-b", "trung-dev", "https://github.com/MinhTuan2405/ViAmpleHate.git"], check=True)
    os.chdir(ROOT / "ViAmpleHate")

ROOT = Path.cwd()
EXP_DIR = ROOT / EXP_REL
SCRIPT = str(EXP_DIR / "run_two_models.py")
REQS = str(EXP_DIR / "requirements.txt")

assert Path(SCRIPT).exists(), f"Không tìm thấy {SCRIPT}"
assert Path(REQS).exists(), f"Không tìm thấy {REQS}"
print("Repo path:", ROOT)
print("Baseline path:", EXP_DIR)


In [ ]:
!pip install -q -r "{REQS}"

## Cấu hình

Notebook này cố định chạy `VOZ-HSD`. Giống các baseline VOZ-HSD gốc: lấy mẫu phân tầng 100.000 dòng, giữ tỉ lệ lớp tự nhiên và chia train/dev/test 80/10/10.

Hai model dùng cùng dataset, split, seed và max length. Training budget được ghi rõ: XLM-RoBERTa-base chạy 5 epochs; ViCLSR chạy 2 epochs với FP16 và batch size 4 vì backbone XLM-RoBERTa-Large nặng hơn đáng kể trên Kaggle. Đây là so sánh baseline theo giới hạn tài nguyên, không phải compute-matched.


In [ ]:
DATASET = "vozhsd"
XLMR_EPOCHS = 5
VICLSR_EPOCHS = 2
MAX_LEN = 128
OUTPUT_DIR = "/kaggle/working/viamplehate_runs_vozhsd_seed42"

# Khớp cách lấy mẫu trong các notebook baseline VOZ-HSD của repo.
VOZ_SPLIT_POLICY = "baseline"
VOZ_SAMPLE_SIZE = 100_000
VOZ_HATE_RATIO = 0.10  # Chỉ được dùng khi policy="proposed".


## Smoke Test

Chạy 2 cell này trước để kiểm tra path, dataset, model loading, forward/backward, save metrics. XLM-RoBERTa smoke test nhanh; ViCLSR smoke test vẫn phải tải checkpoint lớn lần đầu.

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs 1 \
  --max-len 64 \
  --batch-size 2 \
  --eval-batch-size 4 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model viclsr \
  --epochs 1 \
  --max-len 64 \
  --batch-size 1 \
  --eval-batch-size 2 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir /kaggle/working/viamplehate_smoke \
  --smoke-test

## 1. Chạy XLM-RoBERTa

Cấu hình full run: 5 epochs, batch size 8, max length 128.


In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs {XLMR_EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 8 \
  --eval-batch-size 16 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}


## 2. Chạy ViCLSR

Cấu hình full run: 2 epochs, FP16, batch size 4, max length 128. ViCLSR dùng XLM-RoBERTa-Large nên training budget thấp hơn XLM-RoBERTa-base để phù hợp giới hạn thời gian/VRAM Kaggle.


In [ ]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model viclsr \
  --epochs {VICLSR_EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 4 \
  --eval-batch-size 8 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR} \
  --fp16


## 3. Xem metrics

Mỗi model lưu `best_model.pt`, `metrics.json`, và tokenizer vào `/kaggle/working`.

In [ ]:
import json
from pathlib import Path

base = Path(OUTPUT_DIR) / DATASET
for model_name in ["xlm-roberta", "viclsr"]:
    path = base / model_name / "metrics.json"
    print("\n===", model_name, "===")
    if not path.exists():
        print("Chưa có metrics:", path)
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print({k: metrics[k] for k in ["accuracy", "macro_f1", "hate_f1"]})
    print(metrics["report"])

## Ghi chú báo cáo

Ưu tiên so sánh `macro_f1` và `hate_f1` vì class `HATE` ít hơn nhiều so với `NON-HATE`. Báo cáo phải nêu XLM-RoBERTa chạy 5 epochs và ViCLSR chạy 2 epochs; không mô tả đây là so sánh cùng training budget. File `metrics.json` lưu cấu hình thực tế trong trường `config`.
